# Exercise 04 — Tool Use: Reminder Bot

This project teaches you how to give Claude tools so it can access real-world information and take actions.

**What we're building:** A reminder bot that lets users say things like _"remind me about my dentist appointment a week from Thursday"_ and Claude will actually set the reminder — not just talk about it.

**The three problems tools solve:**
1. Claude doesn't know the exact current time
2. Claude sometimes miscalculates date arithmetic
3. Claude can describe but not actually create reminders

**The three tools we'll build:**
1. `get_current_datetime` — what time is it right now?
2. `add_duration_to_datetime` — add a time offset to a date
3. `set_reminder` — actually save the reminder

In [ ]:
import json
from datetime import datetime
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv
import anthropic
from anthropic.types import ToolParam

load_dotenv(dotenv_path="../.env")
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-5"

# In-memory reminders store (in a real app this would be a database)
reminders: list[dict] = []

print("Ready!")

## Part 1 — Tool Functions

Tool functions are plain Python functions. Best practices:
- Use descriptive names for the function AND its arguments
- Validate inputs immediately and raise errors with helpful messages
- Claude can see the error message, so make it specific

In [ ]:
def get_current_datetime(date_format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """Return the current date and time as a formatted string."""
    if not date_format:
        raise ValueError("date_format cannot be empty. Use a strftime format string like '%Y-%m-%d'.")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(
    datetime_str: str,
    years: int = 0,
    months: int = 0,
    days: int = 0,
    hours: int = 0,
    minutes: int = 0
) -> str:
    """Add a time duration to a datetime string and return the result."""
    if not datetime_str:
        raise ValueError("datetime_str cannot be empty.")
    try:
        dt = datetime.strptime(datetime_str, "%Y-%m-%d %H:%M:%S")
    except ValueError:
        raise ValueError(f"datetime_str must be in format '%Y-%m-%d %H:%M:%S', got: {datetime_str!r}")

    result = dt + relativedelta(
        years=years, months=months, days=days,
        hours=hours, minutes=minutes
    )
    return result.strftime("%Y-%m-%d %H:%M:%S")


def set_reminder(reminder_text: str, reminder_datetime: str) -> str:
    """Save a reminder to the in-memory store."""
    if not reminder_text:
        raise ValueError("reminder_text cannot be empty.")
    if not reminder_datetime:
        raise ValueError("reminder_datetime cannot be empty.")

    reminder = {
        "id":       len(reminders) + 1,
        "text":     reminder_text,
        "datetime": reminder_datetime,
    }
    reminders.append(reminder)
    return json.dumps({"success": True, "reminder_id": reminder["id"], "reminder": reminder})


# Quick function tests
now = get_current_datetime()
print(f"Current datetime:  {now}")

future = add_duration_to_datetime(now, days=10, hours=2)
print(f"10 days 2 hrs later: {future}")

result = set_reminder("Take vitamins", future)
print(f"Reminder set: {result}")

## Part 2 — Tool Schemas

A tool schema is a JSON specification that tells Claude:
- What the tool is called
- What it does (description)
- What arguments it takes (input_schema)

Write 3–4 sentences in the description explaining what the tool does, when to use it, and what it returns.

In [ ]:
get_current_datetime_schema = ToolParam(
    name="get_current_datetime",
    description=(
        "Returns the current local date and time as a formatted string. "
        "Use this whenever you need to know what the current date or time is. "
        "The default format is '%Y-%m-%d %H:%M:%S'. "
        "Returns a string like '2025-06-15 14:30:00'."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "strftime format string. Defaults to '%Y-%m-%d %H:%M:%S'."
            }
        },
        "required": []
    }
)

add_duration_to_datetime_schema = ToolParam(
    name="add_duration_to_datetime",
    description=(
        "Adds a time duration to a given datetime string and returns the result. "
        "Use this to calculate future or past dates (e.g. 'one week from Thursday', '3 months from now'). "
        "The datetime_str must be in '%Y-%m-%d %H:%M:%S' format. "
        "Returns the result datetime as a '%Y-%m-%d %H:%M:%S' string."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "datetime_str": {"type": "string", "description": "Base datetime in '%Y-%m-%d %H:%M:%S' format."},
            "years":        {"type": "integer", "description": "Number of years to add."},
            "months":       {"type": "integer", "description": "Number of months to add."},
            "days":         {"type": "integer", "description": "Number of days to add."},
            "hours":        {"type": "integer", "description": "Number of hours to add."},
            "minutes":      {"type": "integer", "description": "Number of minutes to add."}
        },
        "required": ["datetime_str"]
    }
)

set_reminder_schema = ToolParam(
    name="set_reminder",
    description=(
        "Saves a reminder with a text message and a datetime for when it should fire. "
        "Use this after calculating the target datetime to actually create the reminder. "
        "Returns JSON confirming the reminder was saved along with its assigned ID."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "reminder_text":     {"type": "string", "description": "The reminder message text."},
            "reminder_datetime": {"type": "string", "description": "When to fire the reminder ('%Y-%m-%d %H:%M:%S')."}
        },
        "required": ["reminder_text", "reminder_datetime"]
    }
)

ALL_TOOLS = [get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema]

print(f"Defined {len(ALL_TOOLS)} tool schemas.")

## Part 3 — The Tool Dispatcher

When Claude sends a `tool_use` block, we need a function that:
1. Reads the `tool_name` and `tool_input`
2. Calls the right Python function
3. Returns the result (or error string if it fails)

In [ ]:
def run_tool(tool_name: str, tool_input: dict) -> tuple[str, bool]:
    """Dispatch a tool call and return (result_string, is_error)."""
    try:
        if tool_name == "get_current_datetime":
            result = get_current_datetime(**tool_input)
        elif tool_name == "add_duration_to_datetime":
            result = add_duration_to_datetime(**tool_input)
        elif tool_name == "set_reminder":
            result = set_reminder(**tool_input)
        else:
            raise ValueError(f"Unknown tool: {tool_name!r}")
        return json.dumps(result), False
    except Exception as e:
        return str(e), True


def run_tools(assistant_message) -> list[dict]:
    """Process all tool_use blocks in an assistant message. Returns tool_result blocks."""
    tool_results = []
    for block in assistant_message.content:
        if block.type != "tool_use":
            continue
        result_content, is_error = run_tool(block.name, block.input)
        tool_results.append({
            "type":        "tool_result",
            "tool_use_id": block.id,
            "content":     result_content,
            "is_error":    is_error,
        })
    return tool_results


print("Tool dispatcher ready.")

## Part 4 — The Conversation Loop

The conversation loop keeps calling Claude until there are no more tool requests:

```
User message
    → Claude responds with text + tool_use blocks
    → We execute the tools
    → We send back tool_result blocks
    → Claude responds again (possibly with more tools)
    → ... repeat until stop_reason != "tool_use"
    → Return final text response
```

In [ ]:
def text_from_message(message) -> str:
    """Extract and join all text blocks from a Claude message."""
    return " ".join(
        block.text for block in message.content
        if hasattr(block, "text")
    )


def run_conversation(user_input: str, verbose: bool = True) -> str:
    """Run a full tool-enabled conversation and return Claude's final text response."""
    messages = [{"role": "user", "content": user_input}]

    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=ALL_TOOLS,
            messages=messages
        )
        messages.append({"role": "assistant", "content": response.content})

        if verbose:
            for block in response.content:
                if hasattr(block, "text") and block.text:
                    print(f"  [Claude text] {block.text}")
                elif block.type == "tool_use":
                    print(f"  [Tool call]   {block.name}({block.input})")

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        if verbose:
            for r in tool_results:
                status = "ERROR" if r["is_error"] else "OK"
                print(f"  [Tool result] {status}: {r['content'][:100]}")

        messages.append({"role": "user", "content": tool_results})

    return text_from_message(response)

## Part 5 — Testing the Reminder Bot

Let's try several natural language requests. Claude should chain multiple tool calls automatically to produce the correct result.

In [ ]:
# Test 1: Simple "days from now" request
reminders.clear()
print("USER: Set a reminder for my dentist appointment in 10 days.")
print()
final_reply = run_conversation("Set a reminder for my dentist appointment in 10 days.")
print()
print(f"FINAL REPLY: {final_reply}")
print(f"\nReminders stored: {reminders}")

In [ ]:
# Test 2: Complex datetime calculation (week from Thursday)
reminders.clear()
print("USER: Remind me to call my bank a week from this Thursday at 10am.")
print()
final_reply = run_conversation("Remind me to call my bank a week from this Thursday at 10am.")
print()
print(f"FINAL REPLY: {final_reply}")
print(f"\nReminders stored: {json.dumps(reminders, indent=2)}")

In [ ]:
# Test 3: Ask what day it is (single tool call)
print("USER: What day of the week is it today?")
print()
final_reply = run_conversation("What day of the week is it today?", verbose=True)
print()
print(f"FINAL REPLY: {final_reply}")

### Exercise 4a — Add a New Tool

Add a `list_reminders` tool that returns all saved reminders. You need to:
1. Write the Python function
2. Write the tool schema
3. Add the schema to `ALL_TOOLS`
4. Add a dispatch case in `run_tool()`

In [ ]:
# Step 1: Write the function
def list_reminders() -> str:
    """TODO: Return all saved reminders as JSON."""
    pass  # implement this


# Step 2: Write the schema
list_reminders_schema = ToolParam(
    name="list_reminders",
    description="TODO: describe this tool",
    input_schema={
        "type": "object",
        "properties": {},
        "required": []
    }
)

# Step 3: Add to ALL_TOOLS
ALL_TOOLS_EXTENDED = ALL_TOOLS + [list_reminders_schema]

# Step 4: Update run_tool (copy and modify the function above)
# Then test:
# final_reply = run_conversation("Show me all my reminders.")  
print("Implement the function above, then uncomment the test line.")

## Summary

- ✅ Wrote three tool functions with input validation
- ✅ Wrote tool schemas describing each tool to Claude
- ✅ Built a `run_tool()` dispatcher
- ✅ Built a `run_conversation()` loop that chains multiple tool calls automatically
- ✅ Claude can now set real reminders from natural language requests

**Key insight:** The tool_use loop runs an **unknown number of times** — Claude decides how many tools it needs. Your loop must handle 1, 2, or 10 chained calls without changes.

**Next:** [05_advanced_tools.ipynb](05_advanced_tools.ipynb) — batch tools, forced structured extraction, and built-in tools